In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
%pip install -q mcp arxiv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 59.1 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -q -U --force-reinstall --no-cache-dir transformers tokenizers
%pip install -q marker-pdf
%pip install -q mcp arxiv langchain-mcp-adapters langchain-groq langgraph \
    chromadb sentence-transformers rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 216.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 272.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 152.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 195.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 400.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 267.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 280.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 363.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 311.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 377.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 319.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [4]:
%%writefile /kaggle/working/arxiv_server.py
import arxiv
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("arxiv-search")         

@mcp.tool()
def search_papers(query: str, max_results: int = 5) -> list[dict]:
    """Search arXiv for papers matching the query. Returns title,
    authors, year, abstract and pdf_url for each result."""
    results = arxiv.Client().results(
        arxiv.Search(query=query, max_results=max_results)
    )
    return [{
        "title": r.title,
        "authors": [a.name for a in r.authors][:3],
        "year": r.published.year,
        "abstract": r.summary[:400],
        "pdf_url": r.pdf_url,
    } for r in results]

if __name__ == "__main__":
    mcp.run(transport="stdio")         

Writing /kaggle/working/arxiv_server.py


In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command="python",
    args=["/kaggle/working/arxiv_server.py"],   # how to launch the server
)

async with stdio_client(server) as (read, write):        # spawns the subprocess
    async with ClientSession(read, write) as session:
        await session.initialize()                        # MCP handshake

        tools = await session.list_tools()                # "what can you do?"
        for t in tools.tools:
            print("TOOL:", t.name, "—", t.description[:60])

        result = await session.call_tool(                 # "do this"
            "search_papers",
            {"query": "corrective retrieval augmented generation",
             "max_results": 3},
        )
        print(result.content[0].text)

TOOL: search_papers — Search arXiv for papers matching the query. Returns title,
 
{
  "title": "AR-RAG: Autoregressive Retrieval Augmentation for Image Generation",
  "authors": [
    "Jingyuan Qi",
    "Zhiyang Xu",
    "Qifan Wang"
  ],
  "year": 2025,
  "abstract": "We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, ",
  "pdf_url": "https://arxiv.org/pdf/2506.06962v3"
}


In [6]:
%pip install -q langchain-mcp-adapters langchain-groq langgraph

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# sanity check — never print the full key
print("key loaded:", os.environ["GROQ_API_KEY"][:6] + "...")

key loaded: gsk_uD...


In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent          # new import — see note below
from langchain_groq import ChatGroq

client = MultiServerMCPClient({
    "arxiv": {"command": "python",
              "args": ["/kaggle/working/arxiv_server.py"],
              "transport": "stdio"}
})
tools = await client.get_tools()

agent = create_agent(
    ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2),
    tools,
    system_prompt=("You are a research assistant. The ONLY tool available is "
                   "search_papers(query, max_results), which searches arXiv. "
                   "Never call any other tool name. If no tool is needed, just answer.")
)

out = await agent.ainvoke({"messages":
    "Use the search_papers tool to find recent papers on self-correcting "
    "retrieval-augmented generation, then summarize the top result."})
print(out["messages"][-1].content)

for attempt in range(3):
    try:
        out = await agent.ainvoke({"messages": "..."})
        break
    except Exception as e:
        if "tool_use_failed" not in str(e) or attempt == 2:
            raise
        print(f"malformed tool call, retrying ({attempt+1})...")

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


The top result is the paper "AR-RAG: Autoregressive Retrieval Augmentation for Image Generation" by Jingyuan Qi, Zhiyang Xu, and Qifan Wang, published in 2025. This paper introduces a new paradigm called Autoregressive Retrieval Augmentation (AR-RAG) that enhances image generation by incorporating nearest neighbor retrievals at the patch level in an autoregressive manner. Unlike prior methods, AR-RAG performs context-aware retrievals at each generation step, allowing for more dynamic and adaptive image generation. The paper can be found at https://arxiv.org/pdf/2506.06962v3.


In [9]:
from pathlib import Path
import shutil

STORE = Path("/kaggle/working/store")
SNAPSHOT = Path("/kaggle/input/agentic-research-store")  # snapshot dataset (later)

def init_store():
    for sub in ["chroma", "pdfs", "cache/s2"]:
        (STORE / sub).mkdir(parents=True, exist_ok=True)

def restore_store():
    if SNAPSHOT.exists():
        shutil.copytree(SNAPSHOT, STORE, dirs_exist_ok=True)
        return "restored from snapshot"
    init_store()
    return "fresh store (no snapshot attached)"

print(restore_store())
print("STORE =", STORE, "| exists:", STORE.exists())

fresh store (no snapshot attached)
STORE = /kaggle/working/store | exists: True


In [10]:
import hashlib, json, datetime
from pathlib import Path

MANIFEST_PATH = STORE / "manifest.json"
CURRENT_EMBEDDER = "BAAI/bge-base-en-v1.5"   # the guard value
MIN_PARSE_CHARS = 500                         # quarantine threshold

# ---------- load / save ----------

def load_manifest() -> dict:
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"embedder": CURRENT_EMBEDDER, "docs": {}}

def save_manifest(m: dict):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2))

# ---------- identity ----------

def compute_doc_id(pdf_path: Path) -> str:
    return hashlib.sha256(pdf_path.read_bytes()).hexdigest()[:16]

# ---------- the guard ----------

def check_embedder(m: dict):
    if m["docs"] and m["embedder"] != CURRENT_EMBEDDER:
        raise RuntimeError(
            f"Index was built with {m['embedder']} but current embedder is "
            f"{CURRENT_EMBEDDER}. Vectors are incompatible - re-index from "
            f"scratch (delete store/chroma and manifest) before continuing.")

# ---------- main entry ----------

def ingest_pdf(pdf_path: Path, parse_fn, chunk_and_index_fn=None) -> str:
    """Returns one of: 'skipped', 'quarantined', 'ingested'.
    parse_fn(pdf_path) -> markdown string  (your marker call)
    chunk_and_index_fn(markdown, record) -> n_chunks  (plugs in at Step 4)
    """
    m = load_manifest()
    check_embedder(m)

    doc_id = compute_doc_id(pdf_path)
    if doc_id in m["docs"]:
        print(f"  skip (already {m['docs'][doc_id]['status']}): {pdf_path.name}")
        return "skipped"

    markdown = parse_fn(pdf_path)

    record = {
        "doc_id": doc_id,
        "filename": pdf_path.name,
        "title": next((l.lstrip("# ").strip() for l in markdown.splitlines()
                       if l.startswith("# ")), pdf_path.stem),
        "ingested_at": datetime.datetime.utcnow().isoformat(),
        "n_chars": len(markdown),
    }

    if len(markdown) < MIN_PARSE_CHARS:
        record["status"] = "quarantined"
        print(f"  QUARANTINED ({len(markdown)} chars): {pdf_path.name}")
    else:
        if chunk_and_index_fn is not None:
            record["n_chunks"] = chunk_and_index_fn(markdown, record)
        record["status"] = "ingested"
        print(f"  ingested: {record['title'][:60]}")

    m["docs"][doc_id] = record
    save_manifest(m)
    return record["status"]

def ingest_folder(folder: Path, parse_fn, chunk_and_index_fn=None):
    results = [ingest_pdf(p, parse_fn, chunk_and_index_fn)
               for p in sorted(folder.glob("*.pdf"))]
    print(f"\n{results.count('ingested')} ingested, "
          f"{results.count('skipped')} skipped, "
          f"{results.count('quarantined')} quarantined")

def print_manifest_summary():
    m = load_manifest()
    print(f"embedder: {m['embedder']}  |  {len(m['docs'])} documents")
    for d in m["docs"].values():
        print(f"  [{d['status']:11s}] {d.get('n_chunks','-'):>4} chunks  "
              f"{d['title'][:55]}")

In [11]:
import urllib.request

papers = {
    "crag.pdf":     "https://arxiv.org/pdf/2401.15884",   # Corrective RAG
    "self_rag.pdf": "https://arxiv.org/pdf/2310.11511",   # Self-RAG
}
for name, url in papers.items():
    urllib.request.urlretrieve(url, STORE / "pdfs" / name)
    print("downloaded", name)

downloaded crag.pdf
downloaded self_rag.pdf


In [12]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

_models = create_model_dict()          # loads once; slow first time

def parse_with_marker(pdf_path: Path) -> str:
    converter = PdfConverter(artifact_dict=_models)
    rendered = converter(str(pdf_path))
    markdown, _, _ = text_from_rendered(rendered)
    return markdown

2026-06-13 04:14:41.205149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781324081.394281     105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781324081.450674     105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781324081.912238     105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781324081.912279     105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781324081.912282     105 computation_placer.cc:177] computation placer alr

In [13]:
md = parse_with_marker(STORE / "pdfs" / "crag.pdf")
print(md[:3000])

Recognizing Text: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it]
Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7e9a95da7880>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error

# Corrective Retrieval Augmented Generation

Shi-Qi Yan<sup>1</sup>\*, Jia-Chen Gu<sup>2</sup>\*, Yun Zhu<sup>3</sup> , Zhen-Hua Ling<sup>1</sup>

National Engineering Research Center of Speech and Language Information Processing, University of Science and Technology of China, Hefei, China Department of Computer Science, University of California, Los Angeles Google DeepMind

yansiki@mail.ustc.edu.cn, gujc@ucla.edu, yunzhu@google.com, zhling@ustc.edu.cn

# Abstract

Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured solely by the parametric knowledge they encapsulate. Although retrieval-augmented generation (RAG) is a practicable complement to LLMs, it relies heavily on the relevance of retrieved documents, raising concerns about how the model behaves if retrieval goes wrong. To this end, we propose the Corrective Retrieval Augmented Generation (CRAG) to improve the robustness of generation. Specifically, a lightweight 

In [14]:
ingest_folder(STORE / "pdfs", parse_with_marker)   # run 1

Recognizing Text: 100%|██████████| 34/34 [00:03<00:00, 10.73it/s]


  ingested: Corrective Retrieval Augmented Generation


/tmp/ipykernel_105/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),
Recognizing Text: 100%|██████████| 62/62 [00:04<00:00, 12.55it/s]


  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

2 ingested, 0 skipped, 0 quarantined


/tmp/ipykernel_105/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


In [15]:
ingest_folder(STORE / "pdfs", parse_with_marker)
print_manifest_summary()

  skip (already ingested): crag.pdf
  skip (already ingested): self_rag.pdf

0 ingested, 2 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  2 documents
  [ingested   ]    - chunks  Corrective Retrieval Augmented Generation
  [ingested   ]    - chunks  SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE 


In [16]:
import re
import hashlib
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
MAX_TOKENS, OVERLAP_TOKENS = 440, 60     # 440 + contextual header (~25) stays under BGE's 512

def ntok(s):
    return len(tok.encode(s, add_special_tokens=False))

def clean_markdown(md: str) -> str:
    md = re.sub(r"!\[.*?\]\(.*?\)", "", md)            # image artifacts
    md = re.sub(r"<span[^>]*>|</span>", "", md)         # marker's anchor spans
    md = re.sub(r"<sup>.*?</sup>", "", md)              # superscript noise
    return md

def classify_section(path: str) -> str:
    p = path.lower()
    for key in ["abstract", "introduction", "related", "method", "approach",
                "experiment", "result", "discussion", "conclusion", "reference",
                "acknowledg", "appendix"]:
        if key in p:
            return key
    return "body"

def split_by_headers(md: str):
    blocks, path, buf = [], ["PREAMBLE"], []
    for line in md.splitlines():
        m = re.match(r"^(#{1,4})\s+(.*)", line)
        if m:
            if buf:
                blocks.append((" > ".join(path), "\n".join(buf).strip()))
                buf = []
            level, title = len(m.group(1)), m.group(2).strip()
            path = path[:level-1] + [title] if level > 1 else [title]
        else:
            buf.append(line)
    if buf:
        blocks.append((" > ".join(path), "\n".join(buf).strip()))
    return [b for b in blocks if b[1]]

def hard_split(p: str) -> list[str]:
    """Split an oversized paragraph at sentence boundaries; token-slice
    anything that still won't fit (tables, equation blocks)."""
    if ntok(p) <= MAX_TOKENS:
        return [p]
    parts, cur, cur_t = [], [], 0
    for sent in re.split(r"(?<=[.!?])\s+", p):
        st = ntok(sent)
        if cur and cur_t + st > MAX_TOKENS:
            parts.append(" ".join(cur))
            cur, cur_t = [], 0
        cur.append(sent)
        cur_t += st
    if cur:
        parts.append(" ".join(cur))
    # fallback for punctuation-free blobs
    final = []
    for part in parts:
        ids = tok.encode(part, add_special_tokens=False)
        if len(ids) <= MAX_TOKENS:
            final.append(part)
        else:
            final += [tok.decode(ids[i:i + MAX_TOKENS])
                      for i in range(0, len(ids), MAX_TOKENS)]
    return final

def chunk_markdown(md: str, doc: dict) -> list[dict]:
    md = clean_markdown(md)
    out = []
    for sec_path, text in split_by_headers(md):
        stype = classify_section(sec_path)
        if stype in ("reference", "acknowledg", "appendix") or sec_path == "PREAMBLE":
            continue
        sec_h = hashlib.md5(sec_path.encode()).hexdigest()[:8]

        paras = [piece for p in text.split("\n\n") if p.strip()
                 for piece in hard_split(p)]

        chunks_here, cur, cur_t = [], [], 0
        for p in paras:
            pt = ntok(p)
            if cur and cur_t + pt > MAX_TOKENS:
                chunks_here.append("\n\n".join(cur))
                tail, t = [], 0
                for q in reversed(cur):
                    if t + ntok(q) > OVERLAP_TOKENS:
                        break
                    tail.insert(0, q)
                    t += ntok(q)
                # shed overlap if the seed itself would bust the budget
                while tail and t + pt > MAX_TOKENS:
                    t -= ntok(tail.pop(0))
                cur, cur_t = tail + [p], t + pt
            else:
                cur.append(p)
                cur_t += pt
        if cur:
            chunks_here.append("\n\n".join(cur))

        for i, c in enumerate(chunks_here):
            if ntok(c) < 30:
                continue
            out.append({
                "id": f'{doc["doc_id"]}::{sec_h}::{i}',
                "text": c,
                "embed_text": f'{doc["title"]} — {sec_path}\n\n{c}',
                "metadata": {"doc_id": doc["doc_id"],
                             "title": doc["title"][:80],
                             "section": sec_path[:80],
                             "section_type": stype,
                             "chunk_index": i},
            })
    return out

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [17]:
from sentence_transformers import SentenceTransformer
import chromadb

emb_model = SentenceTransformer("BAAI/bge-base-en-v1.5")   # GPU auto-detected
Q_PREFIX = "Represent this sentence for searching relevant passages: "

chroma = chromadb.PersistentClient(path=str(STORE / "chroma"))
collection = chroma.get_or_create_collection("papers", metadata={"hnsw:space": "cosine"})

def chunk_and_index(markdown: str, record: dict) -> int:
    chunks = chunk_markdown(markdown, record)
    if not chunks: return 0
    vecs = emb_model.encode([c["embed_text"] for c in chunks],
                            normalize_embeddings=True, show_progress_bar=False)
    collection.add(
        ids=[c["id"] for c in chunks],
        embeddings=vecs.tolist(),
        documents=[c["text"] for c in chunks],        # clean text for the LLM
        metadatas=[c["metadata"] for c in chunks],
    )
    return len(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
import shutil
def reset_index():
    if MANIFEST_PATH.exists(): MANIFEST_PATH.unlink()
    chroma.delete_collection("papers")
    globals()["collection"] = chroma.get_or_create_collection(
        "papers", metadata={"hnsw:space": "cosine"})
    print("index + manifest wiped")

reset_index()
ingest_folder(STORE / "pdfs", parse_with_marker, chunk_and_index)
print_manifest_summary()

index + manifest wiped


Recognizing Text: 100%|██████████| 34/34 [00:03<00:00, 10.65it/s]
/tmp/ipykernel_105/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),
Token indices sequence length is longer than the specified maximum sequence length for this model (688 > 512). Running this sequence through the model will result in indexing errors


  ingested: Corrective Retrieval Augmented Generation


Recognizing Text: 100%|██████████| 62/62 [00:04<00:00, 12.54it/s]
/tmp/ipykernel_105/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

2 ingested, 0 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  2 documents
  [ingested   ]   49 chunks  Corrective Retrieval Augmented Generation
  [ingested   ]   83 chunks  SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE 


In [19]:
import numpy as np
data = collection.get(include=["documents", "metadatas"])
lens = [ntok(d) for d in data["documents"]]
print(f"{len(lens)} chunks | tokens: min {min(lens)}, "
      f"median {int(np.median(lens))}, max {max(lens)}")

# expose the offenders
for d, m in zip(data["documents"], data["metadatas"]):
    if ntok(d) > MAX_TOKENS:
        print("\nOVERSIZED:", m["section"], "|", ntok(d), "tokens")
        print(d[:300])

132 chunks | tokens: min 32, median 287, max 440


In [20]:
MD_CACHE = STORE / "markdown"; MD_CACHE.mkdir(exist_ok=True)

def parse_cached(pdf_path: Path) -> str:
    cache = MD_CACHE / (compute_doc_id(pdf_path) + ".md")
    if cache.exists():
        return cache.read_text()
    md = parse_with_marker(pdf_path)
    cache.write_text(md)
    return md

In [21]:
reset_index()
ingest_folder(STORE / "pdfs", parse_cached, chunk_and_index)
print_manifest_summary()

index + manifest wiped


Recognizing Text: 100%|██████████| 34/34 [00:03<00:00, 10.73it/s]
/tmp/ipykernel_105/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: Corrective Retrieval Augmented Generation


Recognizing Text: 100%|██████████| 62/62 [00:04<00:00, 12.52it/s]
/tmp/ipykernel_105/2827655713.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.datetime.utcnow().isoformat(),


  ingested: SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU

2 ingested, 0 skipped, 0 quarantined
embedder: BAAI/bge-base-en-v1.5  |  2 documents
  [ingested   ]   49 chunks  Corrective Retrieval Augmented Generation
  [ingested   ]   83 chunks  SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE 


In [22]:
data = collection.get(include=["documents", "metadatas"])
lens = [ntok(d) for d in data["documents"]]
print(f"{len(lens)} chunks | min {min(lens)}, median {int(np.median(lens))}, max {max(lens)}")
assert max(lens) <= MAX_TOKENS, "still oversized!"

132 chunks | min 32, median 287, max 440


In [23]:
def smoke(q):
    qv = emb_model.encode(Q_PREFIX + q, normalize_embeddings=True)
    res = collection.query(query_embeddings=[qv.tolist()], n_results=3)
    print("\nQ:", q)
    for doc, m in zip(res["documents"][0], res["metadatas"][0]):
        print("  →", m["title"][:40], "|", m["section"][:45])
        print("    ", doc[:160].replace("\n", " "))

smoke("What corrective actions does CRAG take when retrieval quality is low?")
smoke("How does Self-RAG use reflection tokens to critique its own generation?")


Q: What corrective actions does CRAG take when retrieval quality is low?
  → Corrective Retrieval Augmented Generatio | 1 Introduction
     On account of the above issues, this paper particularly studies the scenarios where the retriever returns inaccurate results. A method named Corrective Retrieva
  → Corrective Retrieval Augmented Generatio | 6 Conclusion & Limitation
     This paper studies the problem where RAG-based approaches are challenged if retrieval goes wrong, thereby exposing inaccurate and misleading knowledge to genera
  → Corrective Retrieval Augmented Generatio | 5 Experiments
     We conducted experiments to extensively demonstrate CRAG's adaptability to RAG-based approaches and its generalizability across both shortand long-form generati

Q: How does Self-RAG use reflection tokens to critique its own generation?
  → SELF-RAG: LEARNING TO RETRIEVE, GENERATE | A SELF-RAG DETAILS > A.1 REFLECTION TOKENS.
     Definitions of reflection tokens. Below, we provide a detail

In [24]:
def show_chunks(title_substr):
    data = collection.get(include=["documents", "metadatas"])
    for cid, d, m in zip(data["ids"], data["documents"], data["metadatas"]):
        if title_substr.lower() in m["title"].lower():
            print(f"{cid}\n  [{m['section_type']}] {m['section'][:60]}\n  {d[:120]}...\n")

show_chunks("Corrective")   # then show_chunks("SELF-RAG")

975aa1fd3c1b6031::eb581f51::0
  [body] Corrective Retrieval Augmented Generation
  Shi-Qi Yan\*, Jia-Chen Gu\*, Yun Zhu , Zhen-Hua Ling

National Engineering Research Center of Speech and Language Inform...

975aa1fd3c1b6031::e353dbe4::0
  [abstract] Abstract
  Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured s...

975aa1fd3c1b6031::8d2142fa::0
  [introduction] 1 Introduction
  Large language models (LLMs) have attracted increasing attention and exhibited impressive abilities to understand instru...

975aa1fd3c1b6031::8d2142fa::1
  [introduction] 1 Introduction
  **Retrieved Documents** Prior research has introduced the retrieval techniques to incorporate the knowledge relevant to ...

975aa1fd3c1b6031::8d2142fa::2
  [introduction] 1 Introduction
   Equal contribution.

The code is available at [github.com/HuskyInSalt/CRAG](https://github.com/HuskyInSalt/CRAG)

a sub...

975aa1fd3c1b6031::8d2142fa::3
  [introduction]

In [25]:
show_chunks("SELF-RAG")

d9eaa1398abac0df::dfd980ba::0
  [body] SELF-RAG: LEARNING TO RETRIEVE, GENERATE, AND CRITIQUE THROU
  Akari Asai† , Zeqiu Wu† , Yizhong Wang†§, Avirup Sil‡ , Hannaneh Hajishirzi†§ †University of Washington §Allen Institute...

d9eaa1398abac0df::f387fb12::0
  [abstract] ABSTRACT
  Despite their remarkable capabilities, large language models (LLMs) often produce responses containing factual inaccurac...

d9eaa1398abac0df::310aec30::0
  [introduction] 1 INTRODUCTION
  State-of-the-art LLMs continue to struggle with factual errors [\(Mallen et al., 2023;](#page-12-0) [Min et al., 2023\)]...

d9eaa1398abac0df::310aec30::1
  [introduction] 1 INTRODUCTION
  Reflection tokens are categorized into *retrieval* and *critique* tokens to indicate the need for retrieval and its gene...

d9eaa1398abac0df::310aec30::2
  [introduction] 1 INTRODUCTION
  SELF-RAG trains an arbitrary LM to generate text with reflection tokens by unifying them as the next token prediction fr...

d9eaa1398abac0df::310aec30:

In [26]:
C = "975aa1fd3c1b6031"   # CRAG
S = "d9eaa1398abac0df"   # Self-RAG

EVAL_SET = [
  # ---------- factoid ----------
  {"q": "What three retrieval quality judgments does CRAG's evaluator produce?",
   "relevant_ids": [f"{C}::c9d67925::0", f"{C}::c9d67925::1"], "answerable": True},
  {"q": "What corrective action does CRAG trigger when retrieval is judged Incorrect?",
   "relevant_ids": [f"{C}::c9d67925::1", f"{C}::aff5dedc::0"], "answerable": True},
  {"q": "What is the decompose-then-recompose algorithm in CRAG used for?",
   "relevant_ids": [f"{C}::98392ba7::0"], "answerable": True},
  {"q": "What model is CRAG's retrieval evaluator fine-tuned from?",
   "relevant_ids": [f"{C}::8fabe20f::0", f"{C}::0f322ed5::0"], "answerable": True},
  {"q": "What are the four types of reflection tokens in Self-RAG?",
   "relevant_ids": [f"{S}::1f63c8bd::2", f"{S}::e4c534c1::0", f"{S}::e4c534c1::1"],
   "answerable": True},
  {"q": "How is the critic model in Self-RAG trained and where does its training data come from?",
   "relevant_ids": [f"{S}::906aa159::0", f"{S}::906aa159::2"], "answerable": True},
  # ---------- synthesis ----------
  {"q": "What role does web search play in CRAG's pipeline?",
   "relevant_ids": [f"{C}::aff5dedc::0", f"{C}::0198e4dc::0"], "answerable": True},
  {"q": "How does Self-RAG use reflection tokens to control generation at inference time?",
   "relevant_ids": [f"{S}::93ac9dd8::1", f"{S}::30c1c079::0", f"{S}::310aec30::1"],
   "answerable": True},
  {"q": "How do CRAG and Self-RAG differ in how they judge whether retrieval was good?",
   "relevant_ids": [f"{C}::0f322ed5::0", f"{S}::310aec30::1", f"{S}::93ac9dd8::1"],
   "answerable": True},
  # ---------- unanswerable ----------
  {"q": "What does the RAPTOR paper propose for hierarchical summarization?",
   "relevant_ids": [], "answerable": False},
  {"q": "How does GraphRAG build community summaries over a document corpus?",
   "relevant_ids": [], "answerable": False},
  {"q": "What dollar cost did the CRAG authors report for their GPT-4 API usage?",
   "relevant_ids": [], "answerable": False},
]
print(len(EVAL_SET), "questions,",
      sum(e["answerable"] for e in EVAL_SET), "answerable")

12 questions, 9 answerable


In [27]:
def dense_retrieve(q, top_k=5):
    qv = emb_model.encode(Q_PREFIX + q, normalize_embeddings=True)
    res = collection.query(query_embeddings=[qv.tolist()], n_results=top_k)
    return res["ids"][0]

def evaluate(retrieve_fn, k=5, label=""):
    hits, mrrs = [], []
    for ex in EVAL_SET:
        if not ex["answerable"]:
            continue
        ids = retrieve_fn(ex["q"], top_k=k)
        rel = set(ex["relevant_ids"])
        hits.append(float(any(i in rel for i in ids)))
        rank = next((j + 1 for j, i in enumerate(ids) if i in rel), None)
        mrrs.append(1 / rank if rank else 0.0)
    print(f"{label:30s} hit@{k}: {np.mean(hits):.3f}   MRR: {np.mean(mrrs):.3f}")
    return np.mean(hits), np.mean(mrrs)

In [28]:
evaluate(dense_retrieve, k=5, label="dense (BGE + struct chunks)")

dense (BGE + struct chunks)    hit@5: 0.889   MRR: 0.587


(np.float64(0.8888888888888888), np.float64(0.587037037037037))

In [29]:
def show_misses(retrieve_fn, k=5):
    for ex in EVAL_SET:
        if not ex["answerable"]:
            continue
        ids = retrieve_fn(ex["q"], top_k=k)
        rel = set(ex["relevant_ids"])
        rank = next((j+1 for j, i in enumerate(ids) if i in rel), None)
        print(f"rank {rank if rank else 'MISS':>4} | {ex['q'][:70]}")

show_misses(dense_retrieve)

rank    5 | What three retrieval quality judgments does CRAG's evaluator produce?
rank    1 | What corrective action does CRAG trigger when retrieval is judged Inco
rank MISS | What is the decompose-then-recompose algorithm in CRAG used for?
rank    1 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank    2 | How is the critic model in Self-RAG trained and where does its trainin
rank    1 | What role does web search play in CRAG's pipeline?
rank    3 | How does Self-RAG use reflection tokens to control generation at infer
rank    4 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [30]:
import re
from rank_bm25 import BM25Okapi

# pull corpus from Chroma (needs `collection` from Cell B above)
data = collection.get(include=["documents"])
chunk_ids, docs = data["ids"], data["documents"]
doc_lookup = dict(zip(chunk_ids, docs))

def bm25_tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

bm25 = BM25Okapi([bm25_tok(d) for d in docs])

def hybrid_retrieve(q, top_k=5, k_each=20):
    qv = emb_model.encode(Q_PREFIX + q, normalize_embeddings=True)
    dense = collection.query(query_embeddings=[qv.tolist()], n_results=k_each)["ids"][0]
    scores = bm25.get_scores(bm25_tok(q))
    sparse = [chunk_ids[i] for i in
              sorted(range(len(scores)), key=lambda i: -scores[i])[:k_each]]
    K, fused = 60, {}
    for lst in (dense, sparse):
        for r, cid in enumerate(lst):
            fused[cid] = fused.get(cid, 0) + 1/(K + r + 1)
    return sorted(fused, key=fused.get, reverse=True)[:top_k]

In [31]:
evaluate(hybrid_retrieve, k=5, label="hybrid (fixed tokenizer)")
show_misses(hybrid_retrieve)

hybrid (fixed tokenizer)       hit@5: 0.778   MRR: 0.722
rank MISS | What three retrieval quality judgments does CRAG's evaluator produce?
rank    2 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    1 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    1 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank    1 | How is the critic model in Self-RAG trained and where does its trainin
rank    1 | What role does web search play in CRAG's pipeline?
rank    1 | How does Self-RAG use reflection tokens to control generation at infer
rank MISS | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [32]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank_retrieve(q, top_k=5):
    cands = hybrid_retrieve(q, top_k=20)
    scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]

def rerank_dense(q, top_k=5):
    cands = dense_retrieve(q, top_k=20)
    scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]
for name in ["dense_retrieve", "hybrid_retrieve", "rerank_retrieve",
             "rerank_dense", "evaluate", "show_misses", "EVAL_SET"]:
    assert name in globals(), f"MISSING: {name}"
print("spine complete ✓")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

spine complete ✓


In [33]:
evaluate(rerank_retrieve, k=5, label="hybrid + cross-encoder rerank")
show_misses(rerank_retrieve)

hybrid + cross-encoder rerank  hit@5: 0.889   MRR: 0.491
rank    2 | What three retrieval quality judgments does CRAG's evaluator produce?
rank    2 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    4 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    3 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank MISS | How is the critic model in Self-RAG trained and where does its trainin
rank    2 | What role does web search play in CRAG's pipeline?
rank    1 | How does Self-RAG use reflection tokens to control generation at infer
rank    3 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [34]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank_dense(q, top_k=5):
    cands = dense_retrieve(q, top_k=20)
    scores = reranker.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]

evaluate(rerank_dense, k=5, label="dense + rerank (no BM25)")
show_misses(rerank_dense)

dense + rerank (no BM25)       hit@5: 0.889   MRR: 0.593
rank    2 | What three retrieval quality judgments does CRAG's evaluator produce?
rank    2 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    2 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    3 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank MISS | How is the critic model in Self-RAG trained and where does its trainin
rank    2 | What role does web search play in CRAG's pipeline?
rank    1 | How does Self-RAG use reflection tokens to control generation at infer
rank    1 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [35]:
def inspect_top(retrieve_fn, q, k=3):
    print("Q:", q)
    for cid in retrieve_fn(q, top_k=k):
        r = collection.get(ids=[cid], include=["documents","metadatas"])
        print("  •", r["metadatas"][0]["section"][:55], "\n   ", 
              r["documents"][0][:180].replace("\n"," "), "\n")

inspect_top(rerank_retrieve, EVAL_SET[0]["q"])   # three judgments
inspect_top(rerank_retrieve, EVAL_SET[3]["q"])   # T5-large
inspect_top(rerank_retrieve, EVAL_SET[5]["q"])   # critic training (the MISS)

Q: What three retrieval quality judgments does CRAG's evaluator produce?
  • A Task Prompts > B Experiments > B.1 Tasks, Datasets an 
    CRAG was evaluated on four datasets, which are in public domain and licensed for research purposes, including:  PopQA [\(Mallen et al.,](#page-10-2) [2023\)](#page-10-2) is a *shor 

  • 3 Task Formulation > 4.2 Retrieval Evaluator > 4.3 Acti 
    Ambiguous Except for the above two situations, the remaining will be assigned to an intermediate action of Ambiguous. This generally occurs when the accuracy of the retrieval is ha 

  • Abstract 
    Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured solely by the parametric knowledge they encapsulate. Although 

Q: What model is CRAG's retrieval evaluator fine-tuned from?
  • Abstract 
    Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts cannot be secured solely by the parametric knowled

In [36]:
reranker2 = CrossEncoder("BAAI/bge-reranker-v2-m3")   # ~2GB download

def rerank2_retrieve(q, top_k=5):
    cands = hybrid_retrieve(q, top_k=20)
    scores = reranker2.predict([(q, doc_lookup[c]) for c in cands])
    return [c for c, s in sorted(zip(cands, scores), key=lambda x: -x[1])[:top_k]]

evaluate(rerank2_retrieve, k=5, label="hybrid + bge-reranker-v2-m3")
show_misses(rerank2_retrieve)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

hybrid + bge-reranker-v2-m3    hit@5: 0.778   MRR: 0.417
rank MISS | What three retrieval quality judgments does CRAG's evaluator produce?
rank    4 | What corrective action does CRAG trigger when retrieval is judged Inco
rank    3 | What is the decompose-then-recompose algorithm in CRAG used for?
rank    1 | What model is CRAG's retrieval evaluator fine-tuned from?
rank    1 | What are the four types of reflection tokens in Self-RAG?
rank MISS | How is the critic model in Self-RAG trained and where does its trainin
rank    3 | What role does web search play in CRAG's pipeline?
rank    3 | How does Self-RAG use reflection tokens to control generation at infer
rank    2 | How do CRAG and Self-RAG differ in how they judge whether retrieval wa


In [37]:
evaluate(hybrid_retrieve, k=5, label="sanity: which hybrid is live?")

sanity: which hybrid is live?  hit@5: 0.778   MRR: 0.722


(np.float64(0.7777777777777778), np.float64(0.7222222222222222))

In [38]:
%pip install -q langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [39]:
from langchain_groq import ChatGroq
import json, re as _re

fast = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
big  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

GRADE_PROMPT = """Question: {question}

Retrieved passages:
{chunks}

Judge the SET of passages as a whole. Passages being on-topic is NOT enough --
they must actually contain the information needed to answer THIS question.
- "sufficient": the passages contain the information needed to answer fully
- "insufficient": on-topic but missing key parts of the answer
- "irrelevant": mostly off-topic for this question
Return ONLY JSON: {{"grade": "...", "missing": "what is missing, if anything"}}"""

def parse_json_loose(text):
    m = _re.search(r"\{.*\}", text, _re.DOTALL)
    return json.loads(m.group()) if m else {"grade": "PARSE_FAIL", "missing": text[:120]}

def grade(question, model=fast, show_chunks=False):
    ids = rerank_retrieve(question, top_k=5)
    chunks = "\n\n".join(f"[{i+1}] {doc_lookup[c][:600]}" for i, c in enumerate(ids))
    out = model.invoke(GRADE_PROMPT.format(question=question, chunks=chunks))
    verdict = parse_json_loose(out.content)
    print(f"Q: {question}\n  → grade: {verdict['grade']}\n  → missing: {verdict['missing']}\n")
    if show_chunks:
        for i, c in enumerate(ids):
            print(f"  [{i+1}]", doc_lookup[c][:100].replace("\n", " "))
    return verdict

In [40]:
test_cases = [
    # expect: sufficient (answerable, chunks should contain the answer)
    ("What are the four types of reflection tokens in Self-RAG?",        "sufficient"),
    ("What role does web search play in CRAG's pipeline?",               "sufficient"),
    # expect: irrelevant (corpus has NOTHING on these topics)
    ("What does the RAPTOR paper propose for hierarchical summarization?", "irrelevant"),
    ("How does GraphRAG build community summaries over a document corpus?", "irrelevant"),
    # expect: insufficient (partially covered — datasets yes, hyperparams no)
    ("What datasets did CRAG use, and what were the exact training "
     "hyperparameters of Self-RAG's critic on each of them?",            "insufficient"),
    # the hard case: answerable but the method chunk may rank low
    ("What three retrieval quality judgments does CRAG's evaluator produce?", "sufficient"),
]

results = []
for q, expected in test_cases:
    v = grade(q)
    results.append((q[:55], expected, v["grade"],
                    "✓" if v["grade"] == expected else "✗"))

print(f"\n{'question':57s} {'expected':14s} {'got':14s} {'match'}")
print("-" * 92)
for r in results:
    print(f"{r[0]:57s} {r[1]:14s} {r[2]:14s} {r[3]}")

Q: What are the four types of reflection tokens in Self-RAG?
  → grade: sufficient
  → missing: specific details about the types of reflection tokens

Q: What role does web search play in CRAG's pipeline?
  → grade: sufficient
  → missing: specific details about the role of web search in CRAG's pipeline

Q: What does the RAPTOR paper propose for hierarchical summarization?
  → grade: sufficient
  → missing: none

Q: How does GraphRAG build community summaries over a document corpus?
  → grade: insufficient
  → missing: information about GraphRAG's community summary generation process

Q: What datasets did CRAG use, and what were the exact training hyperparameters of Self-RAG's critic on each of them?
  → grade: insufficient
  → missing: the exact training hyperparameters of Self-RAG's critic on each of the datasets used by CRAG

Q: What three retrieval quality judgments does CRAG's evaluator produce?
  → grade: sufficient
  → missing: none


question                                    

In [41]:
print("=== 70B on the failures ===\n")
grade("What does the RAPTOR paper propose for hierarchical summarization?",
      model=big, show_chunks=True)
grade("How does GraphRAG build community summaries over a document corpus?",
      model=big, show_chunks=True)

=== 70B on the failures ===

Q: What does the RAPTOR paper propose for hierarchical summarization?
  → grade: irrelevant
  → missing: The RAPTOR paper and its proposal for hierarchical summarization are not mentioned in the provided passages.

  [1] Despite their remarkable capabilities, large language models (LLMs) often produce responses containi
  [2] Concurrent RAG work. A few concurrent works[2](#page-2-0) on RAG propose new training or prompting s
  [3] Large language models (LLMs) inevitably exhibit hallucinations since the accuracy of generated texts
  [4] Word embedding is the collective name for a set of language modeling and feature learning techniques
  [5] This work introduces Self-Rag, a new framework to enhance the quality and factuality of LLMs through
Q: How does GraphRAG build community summaries over a document corpus?
  → grade: irrelevant
  → missing: Information about how GraphRAG builds community summaries over a document corpus

  [1] Retrieval setup details. By

{'grade': 'irrelevant',
 'missing': 'Information about how GraphRAG builds community summaries over a document corpus'}